# 03 -Evaluation Analysis
Analisis hasil RAGAS evaluation - faithfulness, answer relevancy, context precision.

In [1]:
import sys
from dotenv import load_dotenv

sys.path.insert(0, '../src')
load_dotenv()

True

## 1. Load Pipeline Components

In [2]:
from core.services.rag.vector_store import load_vector_store
from core.services.rag.bm25_retriever import load_bm25
from core.services.rag.embedder import load_embedding_model
from core.services.rag.reranker import load_reranker
from core.services.llm.model_factory import get_llm

VECTOR_DIR = '../storage/vectordb'

faiss_index, chunks = load_vector_store(VECTOR_DIR)
bm25 = load_bm25(VECTOR_DIR)
embedding_model = load_embedding_model()
reranker = load_reranker()
llm = get_llm()

print("All pipeline components loaded successfully")
print(f"LLM backend: {llm.model_name}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All pipeline components loaded successfully
LLM backend: llama-3.1-8b-instant


## 2. Define Test Cases

In [3]:
test_cases = [
    # {
    #     "question": "What are the iron requirements for 1 6-month-old baby?",
    #     "ground_truth": "Infacts 6-12 months require approximately 11,g of iron per day according to WHO guidelines."
    # },
    # {
    #     "question": "When should complementary feeding (MPASI) be introduced?",
    #     "ground_truth": "Complementary feeding should be introfuced at 6 months of age alongside continued breastfeeding."
    # },
    {
        "question": "What are the main causes of stunting in children?",
        "ground_truth": "Stunting is caused by chronic malnutrition, inadequate dietary intake, and repeated infections during the first 1000 days if life."
    },
    # {
    #     "question": "How long should exclusive breastfeeding last?",
    #     "ground_truth": "Exclusive breastfeeding is recommended for the first 6 months of life, followed by continued breastfeeding with complementary foods."
    # },
    # {
    #     "question": "What vitamins are important for child growth?",
    #     "ground_truth": "Vitamin A, Vitamin D, and Vitamin C are critical for child growth, immune function, and bone development."
    # }
]
print(f"Test Cases define: {len(test_cases)}")

Test Cases define: 1


## 3. Run Full Evaluation

In [5]:
from core.services.evaluation.ragas_pipeline import run_full_evaluation

print("Running RAGAS evaluation...")

scores = run_full_evaluation(
    test_cases=test_cases,
    chunks=chunks,
    faiss_index=faiss_index,
    bm25=bm25,
    embedding_model=embedding_model,
    reranker=reranker,
    llm=llm
)

print("\nEvaluation complete!")
print(f"Faithfulness: {scores['faithfulness']}")
print(f"Answer Relevancy: {scores['answer_relevancy']}")
print(f"Context Precision: {scores['context_precision']}")
print(f"Samples evaluated: {scores['n_samples']}")

Running RAGAS evaluation...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]


Evaluation complete!
Faithfulness: 1.0
Answer Relevancy: 0.0
Context Precision: 0.3333
Samples evaluated: 1


## 4. Generate Report

In [6]:
from core.services.evaluation.report_generator import generate_report

report_path = generate_report(scores, output_dir='../storage/evaluation')
print(f"Report saved to: {report_path}")

Report saved to: ..\storage\evaluation\evaluation_report_20260315_142635.json


## 5. Interpret Results

In [7]:
from core.services.evaluation.report_generator import _interpret_scores

interpretation = _interpret_scores(scores)

print("Score Interpretation:")
print("=" * 50)
for metric, text in interpretation.items():
    print(f"{metric:<25} {text}")

Score Interpretation:
faithfulness              ✅ Good - LLM answer are well-grounded in the documents (score: 1.0)
answer_relevancy          ⚠️ Needs inprovement (score: 0.0)
context_precision         ⚠️ Needs inprovement (score: 0.3333)
